# 05: LoRA (Low-Rank Adaptation) & Parameter-Efficient Fine-Tuning (PEFT)

**Track 13: Generative AI, LLMs, RAG & Multi-Agent Swarms** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Mathematics and implementation of LoRA: Low-rank weight update decomposition $W_0 + \frac{\alpha}{r} B A$, parameter footprint reduction, and fine-tuning PyTorch layers.


## 1. LoRA Mathematical Decomposition
$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} B A$$
where $W_0 \in \mathbb{R}^{d \times k}$ is frozen, $A \in \mathbb{R}^{r \times k}$ initialized as $\mathcal{N}(0, \sigma^2)$, and $B \in \mathbb{R}^{d \times r}$ initialized as $0$.

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, lora_alpha=8.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = lora_alpha / rank
        
        self.base_layer = nn.Linear(in_features, out_features)
        self.base_layer.weight.requires_grad = False
        
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) / math.sqrt(in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        
    def forward(self, x):
        base_out = self.base_layer(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

in_dim, out_dim, r = 4096, 4096, 8
lora_layer = LoRALinear(in_dim, out_dim, rank=r)

base_params = in_dim * out_dim
trainable_lora_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)

print("=== LoRA Parameter Efficiency ===")
print(f"Standard Linear Weights  : {base_params:,} parameters")
print(f"Trainable LoRA Weights   : {trainable_lora_params:,} parameters (Rank={r})")
print(f"Parameter Reduction Ratio: {base_params / trainable_lora_params:.1f}x reduction ({(trainable_lora_params/base_params)*100:.3f}% trained)")